# Обучение модели: хищный ли динозавр

Датасет: [Jurassic Park — The Exhaustive Dinosaur Dataset](https://www.kaggle.com/datasets/kjanjua/jurassic-park-the-exhaustive-dinosaur-dataset) (каталог NHM, ~310 родов).

Готовых sklearn-моделей под скейтбординг на Kaggle нет (там видео/CNN). Для инференса в том же FastAPI-каркасе, что и семинар, проще табличная задача: по типу, длине, периоду и региону предсказать, был ли динозавр **carnivorous**.

Качество модели в ДЗ не оценивается. Здесь логистическая регрессия в `sklearn.Pipeline`.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path("..").resolve()
DATA_PATH = Path("dinosaurs.csv")
ARTIFACT_PATH = ROOT / "artifact" / "model.joblib"

FEATURES = ["dino_type", "length_m", "period", "lived_in"]
df_raw = pd.read_csv(DATA_PATH)
df_raw.head()

In [ ]:
df = df_raw.copy()
df["dino_type"] = df["type"].astype(str).str.strip()
df["length_m"] = pd.to_numeric(
    df["length"].astype(str).str.replace("m", "", regex=False),
    errors="coerce",
)
df["period"] = df["period"].fillna("").str.split().str[:2].str.join(" ")
df.loc[df["period"] == "", "period"] = None
df["lived_in"] = df["lived_in"].replace("", pd.NA)
df["carnivorous"] = (df["diet"].str.lower() == "carnivorous").astype(int)
valid_types = {
    "sauropod", "large theropod", "small theropod",
    "euornithopod", "armoured dinosaur", "ceratopsian",
}
df = df.dropna(subset=["dino_type", "diet"])
df = df[df["dino_type"].isin(valid_types)]
df = df[df["period"].fillna("").str.contains("Triassic|Jurassic|Cretaceous", regex=True)]

print(df.shape)
print(df["carnivorous"].value_counts())
print(df[FEATURES].isna().sum())

In [ ]:
numeric = ["length_m"]
categorical = ["dino_type", "period", "lived_in"]

pre = ColumnTransformer(
    [
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical,
        ),
    ]
)

pipeline = Pipeline(
    [
        ("pre", pre),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    df[FEATURES],
    df["carnivorous"],
    test_size=0.2,
    random_state=42,
    stratify=df["carnivorous"],
)
pipeline.fit(X_train, y_train)
proba = pipeline.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
print("accuracy", round(accuracy_score(y_test, pred), 3))
print("roc_auc", round(roc_auc_score(y_test, proba), 3))

In [ ]:
pipeline.fit(df[FEATURES], df["carnivorous"])
ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "pipeline": pipeline,
        "metadata": {
            "model_version": "1.0",
            "features": FEATURES,
            "threshold": 0.5,
            "task": "predict whether a dinosaur is carnivorous",
            "source": "Kaggle / NHM Jurassic Park exhaustive dinosaur dataset",
        },
    },
    ARTIFACT_PATH,
)
print("saved", ARTIFACT_PATH)